# 실습: 데이터 정제(Cleansing) 및 변환(Conversion) (Hands-on Notebook)

이 노트북은 학습자를 위한 **단계별 워크샵(step-by-step workshop)** 형태로 설계되었습니다.

다음 내용을 실습합니다:
1. 데이터 유형 필터링 및 변환 (CSV/DOCX -> JSONL)
2. A/S(After-Sales, 사후 서비스) 로그 정제(cleansing) 및 정규화(normalization)
3. 중복 제거(deduplication) 기법 (완전 일치 + 근사 중복)
4. PII(개인식별정보) 필터링 및 마스킹(masking)
5. 학습(training) 준비가 완료된 최종 JSONL 파일 내보내기(export)

> 이 노트북은 위에서 아래로, 셀을 하나씩 순서대로 실행하세요.

## 0) 환경 설정(Environment Setup)

이 워크샵에서는 일반적인 Python 데이터 처리 도구를 사용합니다:
- 표(tabular) 데이터 변환: `pandas`
- DOCX 읽기/쓰기: `python-docx`
- 정규화 및 내보내기: `json`, `re`, `pathlib`

아래 패키지가 설치되어 있지 않다면, 주석을 해제하고 설치 명령을 실행하세요.

In [1]:
# Uncomment if needed:
%pip install pandas python-docx requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
from pathlib import Path
from datetime import datetime

import pandas as pd

try:
    from docx import Document
except ImportError as exc:
    raise ImportError("python-docx is required. Install with: pip install python-docx") from exc

pd.set_option("display.max_colwidth", 120)

BASE_DIR = Path(".").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

for p in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("기본 디렉터리:", BASE_DIR)
print("생성됨:", RAW_DIR, INTERIM_DIR, PROCESSED_DIR, sep="\n")

기본 디렉터리: /home/ethan/newgen/KMOU_Course/Module_4
생성됨:
/home/ethan/newgen/KMOU_Course/Module_4/data/raw
/home/ethan/newgen/KMOU_Course/Module_4/data/interim
/home/ethan/newgen/KMOU_Course/Module_4/data/processed


## 1) 샘플 데이터셋 가져오기 (A/S 유사 로그 데이터)

공개 **Apache combined (access) log** 샘플(브라우저가 웹 서버에 남기는 로그 라인)을 다운로드한 뒤, **A/S 스타일 synthetic 라인 몇 개를 추가**하여 이후 단계에서 `ticket=` 및 `/service/...` 경로를 추출할 수 있게 합니다.

인터넷 접속이 불가능한 경우, 노트북은 synthetic 대체(fallback) 라인만 기록합니다.

In [3]:
import requests

# Combined (access) log format — loghub's Apache_2k.log is error-log style and will not match the Step 2 parser.
sample_log_url = (
    "https://raw.githubusercontent.com/elastic/examples/master/"
    "Common%20Data%20Formats/apache_logs/apache_logs"
)
raw_log_path = RAW_DIR / "apache_sample.log"

fallback_lines = [
    '203.0.113.10 - - [15/Apr/2026:08:31:20 +0000] "GET /service/aircon?ticket=AS-1001 HTTP/1.1" 200 482 "-" "Mozilla/5.0"',
    '198.51.100.77 - - [15/Apr/2026:08:31:30 +0000] "POST /service/washer?ticket=AS-1002 HTTP/1.1" 500 1210 "-" "Mozilla/5.0"',
    '198.51.100.77 - - [15/Apr/2026:08:31:31 +0000] "POST /service/washer?ticket=AS-1002 HTTP/1.1" 500 1210 "-" "Mozilla/5.0"',
    '192.0.2.20 - - [15/Apr/2026:08:33:40 +0000] "GET /service/fridge?ticket=AS-1003 HTTP/1.1" 404 532 "-" "Mozilla/5.0"',
    '192.0.2.20 - - [15/Apr/2026:08:34:01 +0000] "GET /service/fridge?ticket=AS-1003 HTTP/1.1" 200 900 "-" "Mozilla/5.0"',
    '198.51.100.90 - - [15/Apr/2026:09:00:00 +0000] "GET /service/aircon?ticket=AS-1004 HTTP/1.1" 200 640 "-" "Mozilla/5.0"',
]

download_ok = False
try:
    response = requests.get(sample_log_url, timeout=30)
    response.raise_for_status()
    combined_body = response.text.rstrip("\n")
    supplement = "\n".join(fallback_lines)
    raw_log_path.write_text(combined_body + "\n" + supplement + "\n", encoding="utf-8")
    download_ok = True
except Exception as e:
    raw_log_path.write_text("\n".join(fallback_lines), encoding="utf-8")
    print("다운로드 실패 -> 대체 synthetic 로그를 사용합니다.")
    print("사유:", e)

print("저장된 로그 파일:", raw_log_path)
print("공개 데이터셋 사용:", download_ok)
print("행 수:", len(raw_log_path.read_text(encoding='utf-8', errors='ignore').splitlines()))

저장된 로그 파일: /home/ethan/newgen/KMOU_Course/Module_4/data/raw/apache_sample.log
공개 데이터셋 사용: True
행 수: 10006


## 2) 원시(Raw) 로그를 구조화된 DataFrame으로 파싱

이 단계에서는 비구조화된 로그 라인을 표 형태의 열(column)로 변환합니다:
- `client_ip`
- `timestamp_raw`
- `method`, `endpoint`, `http_version`
- `status_code`, `bytes_sent`

이후 A/S 중심 필드인 `ticket_id`, `service_type`, `event_text`를 생성합니다.

In [4]:
log_pattern = re.compile(
    r'(?P<client_ip>\S+) \S+ \S+ \[(?P<timestamp_raw>[^\]]+)\] "(?P<method>\S+) (?P<endpoint>\S+) (?P<http_version>[^"]+)" (?P<status_code>\d{3}) (?P<bytes_sent>\S+)'
)

records = []
for line in raw_log_path.read_text(encoding="utf-8", errors="ignore").splitlines():
    m = log_pattern.search(line)
    if not m:
        continue
    rec = m.groupdict()
    endpoint = rec["endpoint"]

    ticket_match = re.search(r"ticket=([A-Za-z0-9\-]+)", endpoint)
    service_match = re.search(r"/service/([A-Za-z0-9_\-]+)", endpoint)

    rec["ticket_id"] = ticket_match.group(1) if ticket_match else None
    rec["service_type_raw"] = service_match.group(1) if service_match else None
    rec["event_text"] = f"{rec['method']} {endpoint} -> {rec['status_code']}"
    records.append(rec)

df = pd.DataFrame(records)

if df.empty:
    raise ValueError("No rows parsed. Check regex or source file format.")

print("파싱된 행 수:", len(df))
df.head(8)

파싱된 행 수: 10006


,client_ip,timestamp_raw,method,endpoint,http_version,status_code,bytes_sent,ticket_id,service_type_raw,event_text
0,83.149.9.216,17/May/2015:10:05:03 +0000,GET,/presentations/logstash-monitorama-2013/images/kibana-search.png,HTTP/1.1,200,203023,NaN,NaN,GET /presentations/logstash-monitorama-2013/images/kibana-search.png -> 200
1,83.149.9.216,17/May/2015:10:05:43 +0000,GET,/presentations/logstash-monitorama-2013/images/kibana-dashboard3.png,HTTP/1.1,200,171717,NaN,NaN,GET /presentations/logstash-monitorama-2013/images/kibana-dashboard3.png -> 200
2,83.149.9.216,17/May/2015:10:05:47 +0000,GET,/presentations/logstash-monitorama-2013/plugin/highlight/highlight.js,HTTP/1.1,200,26185,NaN,NaN,GET /presentations/logstash-monitorama-2013/plugin/highlight/highlight.js -> 200
3,83.149.9.216,17/May/2015:10:05:12 +0000,GET,/presentations/logstash-monitorama-2013/plugin/zoom-js/zoom.js,HTTP/1.1,200,7697,NaN,NaN,GET /presentations/logstash-monitorama-2013/plugin/zoom-js/zoom.js -> 200
4,83.149.9.216,17/May/2015:10:05:07 +0000,GET,/presentations/logstash-monitorama-2013/plugin/notes/notes.js,HTTP/1.1,200,2892,NaN,NaN,GET /presentations/logstash-monitorama-2013/plugin/notes/notes.js -> 200
5,83.149.9.216,17/May/2015:10:05:34 +0000,GET,/presentations/logstash-monitorama-2013/images/sad-medic.png,HTTP/1.1,200,430406,NaN,NaN,GET /presentations/logstash-monitorama-2013/images/sad-medic.png -> 200
6,83.149.9.216,17/May/2015:10:05:57 +0000,GET,/presentations/logstash-monitorama-2013/css/fonts/Roboto-Bold.ttf,HTTP/1.1,200,38720,NaN,NaN,GET /presentations/logstash-monitorama-2013/css/fonts/Roboto-Bold.ttf -> 200
7,83.149.9.216,17/May/2015:10:05:50 +0000,GET,/presentations/logstash-monitorama-2013/css/fonts/Roboto-Regular.ttf,HTTP/1.1,200,41820,NaN,NaN,GET /presentations/logstash-monitorama-2013/css/fonts/Roboto-Regular.ttf -> 200


## 3) 데이터 유형 필터링 및 변환

이제 열을 적절한 데이터 유형으로 변환하고 유효하지 않은 행(row)을 필터링합니다:
- 타임스탬프(timestamp)를 `datetime`으로 파싱
- status와 bytes를 숫자(numeric)형으로 변환
- 불가능한 값 제거
- 의미 있는 ticket/service 정보가 있는 행만 유지

In [5]:
work = df.copy()

# 1) Convert timestamp
work["timestamp"] = pd.to_datetime(
    work["timestamp_raw"],
    format="%d/%b/%Y:%H:%M:%S %z",
    errors="coerce",
)

# 2) Convert numeric columns
work["status_code"] = pd.to_numeric(work["status_code"], errors="coerce")
work["bytes_sent"] = pd.to_numeric(work["bytes_sent"].replace("-", pd.NA), errors="coerce")

# 3) Basic filtering rules
before = len(work)
work = work[work["timestamp"].notna()]
work = work[work["status_code"].between(100, 599, inclusive="both")]
work = work[work["service_type_raw"].notna()]
after = len(work)

print(f"필터링 전 행 수: {before}")
print(f"필터링 후 행 수:  {after}")
work[["timestamp", "status_code", "bytes_sent", "ticket_id", "service_type_raw"]].head(8)

필터링 전 행 수: 10006
필터링 후 행 수:  6


,timestamp,status_code,bytes_sent,ticket_id,service_type_raw
10000,2026-04-15 08:31:20+00:00,200,482.0,AS-1001,aircon
10001,2026-04-15 08:31:30+00:00,500,1210.0,AS-1002,washer
10002,2026-04-15 08:31:31+00:00,500,1210.0,AS-1002,washer
10003,2026-04-15 08:33:40+00:00,404,532.0,AS-1003,fridge
10004,2026-04-15 08:34:01+00:00,200,900.0,AS-1003,fridge
10005,2026-04-15 09:00:00+00:00,200,640.0,AS-1004,aircon


## 4) A/S 로그 정제(Cleansing) 및 정규화(Normalization)

이 단계에서는 하위 분석/ML 파이프라인을 신뢰할 수 있도록 비즈니스 필드를 정규화합니다:
- `service_type` 표준화
- 정규화된 `status_label` 추가
- 정규(canonical) 텍스트 필드 구성
- 문자열 노이즈 정리 (공백 trim, 필요 시 소문자화)

In [6]:
service_map = {
    "aircon": "air_conditioner",
    "ac": "air_conditioner",
    "washer": "washing_machine",
    "washingmachine": "washing_machine",
    "fridge": "refrigerator",
    "refrigerator": "refrigerator",
}

clean = work.copy()

clean["service_type"] = (
    clean["service_type_raw"].astype(str).str.strip().str.lower().map(service_map).fillna("other")
)

clean["ticket_id"] = clean["ticket_id"].fillna("UNKNOWN").astype(str).str.strip().str.upper()

clean["status_label"] = pd.cut(
    clean["status_code"],
    bins=[99, 299, 399, 499, 599],
    labels=["success", "redirect", "client_error", "server_error"],
)

clean["event_text_normalized"] = (
    "ticket=" + clean["ticket_id"]
    + " | service=" + clean["service_type"]
    + " | status=" + clean["status_label"].astype(str)
)

clean = clean.sort_values("timestamp").reset_index(drop=True)

clean[[
    "timestamp", "ticket_id", "service_type", "status_code", "status_label", "event_text_normalized"
]].head(10)

,timestamp,ticket_id,service_type,status_code,status_label,event_text_normalized
0,2026-04-15 08:31:20+00:00,AS-1001,air_conditioner,200,success,ticket=AS-1001 | service=air_conditioner | status=success
1,2026-04-15 08:31:30+00:00,AS-1002,washing_machine,500,server_error,ticket=AS-1002 | service=washing_machine | status=server_error
2,2026-04-15 08:31:31+00:00,AS-1002,washing_machine,500,server_error,ticket=AS-1002 | service=washing_machine | status=server_error
3,2026-04-15 08:33:40+00:00,AS-1003,refrigerator,404,client_error,ticket=AS-1003 | service=refrigerator | status=client_error
4,2026-04-15 08:34:01+00:00,AS-1003,refrigerator,200,success,ticket=AS-1003 | service=refrigerator | status=success
5,2026-04-15 09:00:00+00:00,AS-1004,air_conditioner,200,success,ticket=AS-1004 | service=air_conditioner | status=success


## 5) CSV 및 DOCX 입력 파일 생성 (형식 변환 실습용)

변환 파이프라인 실습을 위해 의도적으로 두 가지 소스 형식을 만듭니다:
- `as_logs.csv`
- `as_logs.docx`

DOCX 파일에는 각 이벤트를 key-value 텍스트 한 줄로 저장합니다.

In [7]:
csv_path = INTERIM_DIR / "as_logs.csv"
docx_path = INTERIM_DIR / "as_logs.docx"

csv_cols = [
    "timestamp", "ticket_id", "service_type", "status_code", "status_label", "client_ip", "event_text_normalized"
]
clean[csv_cols].to_csv(csv_path, index=False, encoding="utf-8")

doc = Document()
doc.add_heading("A/S Service Log Notes", level=1)
for _, row in clean.head(200).iterrows():
    line = (
        f"timestamp={row['timestamp']} | ticket_id={row['ticket_id']} | service_type={row['service_type']} | "
        f"status_code={int(row['status_code'])} | status_label={row['status_label']} | "
        f"client_ip={row['client_ip']} | event_text={row['event_text_normalized']}"
    )
    doc.add_paragraph(line)
doc.save(docx_path)

print("생성됨:")
print("-", csv_path)
print("-", docx_path)

생성됨:
- /home/ethan/newgen/KMOU_Course/Module_4/data/interim/as_logs.csv
- /home/ethan/newgen/KMOU_Course/Module_4/data/interim/as_logs.docx


## 6) CSV를 JSONL로 변환

JSONL(JSON Lines)은 각 라인이 독립적인 JSON 레코드이므로 ML 파이프라인에 적합합니다.

CSV를 읽어 행(row)마다 JSON 객체 하나를 기록합니다.

In [8]:
jsonl_from_csv = PROCESSED_DIR / "as_logs_from_csv.jsonl"

csv_df = pd.read_csv(csv_path)

with jsonl_from_csv.open("w", encoding="utf-8") as f:
    for rec in csv_df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("저장됨:", jsonl_from_csv)
print("레코드 수:", len(csv_df))
print("미리보기:")
for i, line in enumerate(jsonl_from_csv.read_text(encoding="utf-8").splitlines()[:2], start=1):
    print(i, line)

저장됨: /home/ethan/newgen/KMOU_Course/Module_4/data/processed/as_logs_from_csv.jsonl
레코드 수: 6
미리보기:
1 {"timestamp": "2026-04-15 08:31:20+00:00", "ticket_id": "AS-1001", "service_type": "air_conditioner", "status_code": 200, "status_label": "success", "client_ip": "203.0.113.10", "event_text_normalized": "ticket=AS-1001 | service=air_conditioner | status=success"}
2 {"timestamp": "2026-04-15 08:31:30+00:00", "ticket_id": "AS-1002", "service_type": "washing_machine", "status_code": 500, "status_label": "server_error", "client_ip": "198.51.100.77", "event_text_normalized": "ticket=AS-1002 | service=washing_machine | status=server_error"}


## 7) DOCX를 JSONL로 변환

다음 스키마를 따르는 DOCX 단락(paragraph)을 파싱합니다:
`key=value | key=value | ...`

이후 JSONL로 직렬화(serialize)합니다.

In [9]:
jsonl_from_docx = PROCESSED_DIR / "as_logs_from_docx.jsonl"

doc = Document(docx_path)
doc_records = []

for para in doc.paragraphs:
    text = para.text.strip()
    if not text or "=" not in text:
        continue
    parts = [p.strip() for p in text.split("|")]
    rec = {}
    for part in parts:
        if "=" not in part:
            continue
        k, v = part.split("=", 1)
        rec[k.strip()] = v.strip()
    if rec:
        doc_records.append(rec)

with jsonl_from_docx.open("w", encoding="utf-8") as f:
    for rec in doc_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("저장됨:", jsonl_from_docx)
print("레코드 수:", len(doc_records))
print("미리보기:")
for i, line in enumerate(jsonl_from_docx.read_text(encoding="utf-8").splitlines()[:2], start=1):
    print(i, line)

저장됨: /home/ethan/newgen/KMOU_Course/Module_4/data/processed/as_logs_from_docx.jsonl
레코드 수: 6
미리보기:
1 {"timestamp": "2026-04-15 08:31:20+00:00", "ticket_id": "AS-1001", "service_type": "air_conditioner", "status_code": "200", "status_label": "success", "client_ip": "203.0.113.10", "event_text": "ticket=AS-1001", "service": "air_conditioner", "status": "success"}
2 {"timestamp": "2026-04-15 08:31:30+00:00", "ticket_id": "AS-1002", "service_type": "washing_machine", "status_code": "500", "status_label": "server_error", "client_ip": "198.51.100.77", "event_text": "ticket=AS-1002", "service": "washing_machine", "status": "server_error"}


## 8) JSONL 소스 병합 및 최종 스키마 정규화

실제 파이프라인에서는 여러 소스 형식을 함께 수집(ingest)하는 경우가 많습니다. 이제 다음을 수행합니다:
- 두 JSONL 파일 읽기
- 필드명 정렬(align)
- 데이터 유형 정규화
- 통합(unified) 데이터셋 생성

In [10]:
def read_jsonl(path: Path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
    return rows

rows_csv = read_jsonl(jsonl_from_csv)
rows_docx = read_jsonl(jsonl_from_docx)

all_rows = rows_csv + rows_docx
unified = pd.DataFrame(all_rows)

# unify a few expected columns
if "event_text" in unified.columns and "event_text_normalized" not in unified.columns:
    unified["event_text_normalized"] = unified["event_text"]

unified["timestamp"] = pd.to_datetime(unified["timestamp"], errors="coerce", utc=True)
unified["status_code"] = pd.to_numeric(unified["status_code"], errors="coerce")
unified["ticket_id"] = unified["ticket_id"].astype(str).str.upper().str.strip()
unified["service_type"] = unified["service_type"].astype(str).str.lower().str.strip()

unified = unified.dropna(subset=["timestamp", "status_code"]).reset_index(drop=True)

print("통합된 행 수:", len(unified))
unified.head(6)

통합된 행 수: 12


,timestamp,ticket_id,service_type,status_code,status_label,client_ip,event_text_normalized,event_text,service,status
0,2026-04-15 08:31:20+00:00,AS-1001,air_conditioner,200,success,203.0.113.10,ticket=AS-1001 | service=air_conditioner | status=success,NaN,NaN,NaN
1,2026-04-15 08:31:30+00:00,AS-1002,washing_machine,500,server_error,198.51.100.77,ticket=AS-1002 | service=washing_machine | status=server_error,NaN,NaN,NaN
2,2026-04-15 08:31:31+00:00,AS-1002,washing_machine,500,server_error,198.51.100.77,ticket=AS-1002 | service=washing_machine | status=server_error,NaN,NaN,NaN
3,2026-04-15 08:33:40+00:00,AS-1003,refrigerator,404,client_error,192.0.2.20,ticket=AS-1003 | service=refrigerator | status=client_error,NaN,NaN,NaN
4,2026-04-15 08:34:01+00:00,AS-1003,refrigerator,200,success,192.0.2.20,ticket=AS-1003 | service=refrigerator | status=success,NaN,NaN,NaN
5,2026-04-15 09:00:00+00:00,AS-1004,air_conditioner,200,success,198.51.100.90,ticket=AS-1004 | service=air_conditioner | status=success,NaN,NaN,NaN


## 9) 중복 제거(Deduplication) 기법

두 가지 방식을 적용합니다:

1. **완전 중복(exact duplicate) 제거**
   - ticket, timestamp, status, event text가 동일한 경우

2. **근사 중복(near duplicate) 제거**
   - 선택한 열로 구성한 정규화 키(normalized key) 기반
   - 공백/대소문자만 다르고 의미는 같은 경우에 유용

In [11]:
dedup = unified.copy()
before = len(dedup)

dedup = dedup.drop_duplicates(
    subset=["ticket_id", "timestamp", "status_code", "event_text_normalized"],
    keep="first",
)
after_exact = len(dedup)

def normalize_text_for_key(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9_=|\- ]", "", text)
    return text

dedup["near_dup_key"] = (
    dedup["ticket_id"].astype(str).str.upper().str.strip()
    + "||"
    + dedup["service_type"].astype(str).str.lower().str.strip()
    + "||"
    + dedup["event_text_normalized"].astype(str).apply(normalize_text_for_key)
)

dedup = dedup.drop_duplicates(subset=["near_dup_key"], keep="first").drop(columns=["near_dup_key"])
after_near = len(dedup)

print(f"중복 제거 전 행 수:         {before}")
print(f"완전 중복 제거 후:         {after_exact}")
print(f"근사 중복 제거 후:{after_near}")

중복 제거 전 행 수:         12
완전 중복 제거 후:         12
근사 중복 제거 후:9


## 10) PII(개인식별정보) 탐지 및 마스킹(Masking)

지원(support) 로그의 자유 텍스트에는 다음과 같은 PII가 포함될 수 있습니다:
- 이메일(Email)
- 전화번호(Phone numbers)
- IP 주소(IP addresses)

내보내기(export) 전에 이러한 값을 탐지하고 마스킹합니다.

In [12]:
# Add synthetic PII examples to demonstrate masking logic.
if len(dedup) > 0:
    dedup.loc[dedup.index[0], "event_text_normalized"] += " | contact=jane.doe@example.com | phone=010-1234-5678"
if len(dedup) > 1:
    dedup.loc[dedup.index[1], "event_text_normalized"] += " | alt_phone=+82 10 9999 1111"

email_re = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
phone_re = re.compile(r"(?:\+?\d{1,3}[\s-]?)?(?:\d{2,4}[\s-]?){2,4}\d{3,4}")
ip_re = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

def mask_pii(text: str) -> str:
    x = str(text)
    x = email_re.sub("[EMAIL_REDACTED]", x)
    x = phone_re.sub("[PHONE_REDACTED]", x)
    x = ip_re.sub("[IP_REDACTED]", x)
    return x

pii = dedup.copy()
pii["event_text_masked"] = pii["event_text_normalized"].apply(mask_pii)
pii["client_ip_masked"] = pii["client_ip"].astype(str).apply(mask_pii)

pii[["event_text_normalized", "event_text_masked", "client_ip", "client_ip_masked"]].head(5)

,event_text_normalized,event_text_masked,client_ip,client_ip_masked
0,ticket=AS-1001 | service=air_conditioner | status=success | contact=jane.doe@example.com | phone=010-1234-5678,ticket=AS-1001 | service=air_conditioner | status=success | contact=[EMAIL_REDACTED] | phone=[PHONE_REDACTED],203.0.113.10,[IP_REDACTED]
1,ticket=AS-1002 | service=washing_machine | status=server_error | alt_phone=+82 10 9999 1111,ticket=AS-1002 | service=washing_machine | status=server_error | alt_phone=[PHONE_REDACTED],198.51.100.77,[IP_REDACTED]
3,ticket=AS-1003 | service=refrigerator | status=client_error,ticket=AS-1003 | service=refrigerator | status=client_error,192.0.2.20,[IP_REDACTED]
4,ticket=AS-1003 | service=refrigerator | status=success,ticket=AS-1003 | service=refrigerator | status=success,192.0.2.20,[IP_REDACTED]
5,ticket=AS-1004 | service=air_conditioner | status=success,ticket=AS-1004 | service=air_conditioner | status=success,198.51.100.90,[IP_REDACTED]


## 11) 학습(Training) 준비 JSONL 최종 내보내기

정제(clean), 정규화(normalized), 중복 제거(deduplicated), 마스킹(masked)된 데이터셋을 내보냅니다.

권장 최종 필드:
- `timestamp`
- `ticket_id`
- `service_type`
- `status_code`
- `status_label`
- `event_text_masked`
- `client_ip_masked`

In [13]:
final_cols = [
    "timestamp", "ticket_id", "service_type", "status_code", "status_label", "event_text_masked", "client_ip_masked"
]

final_df = pii.copy()
final_df["timestamp"] = final_df["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
final_df = final_df[final_cols].reset_index(drop=True)

final_jsonl_path = PROCESSED_DIR / "as_logs_final_cleaned.jsonl"
with final_jsonl_path.open("w", encoding="utf-8") as f:
    for rec in final_df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("최종 JSONL 저장됨:", final_jsonl_path)
print("최종 레코드 수:", len(final_df))
print("출력 샘플:")
for line in final_jsonl_path.read_text(encoding="utf-8").splitlines()[:3]:
    print(line)

최종 JSONL 저장됨: /home/ethan/newgen/KMOU_Course/Module_4/data/processed/as_logs_final_cleaned.jsonl
최종 레코드 수: 9
출력 샘플:
{"timestamp": "2026-04-15T08:31:20Z", "ticket_id": "AS-1001", "service_type": "air_conditioner", "status_code": 200, "status_label": "success", "event_text_masked": "ticket=AS-1001 | service=air_conditioner | status=success | contact=[EMAIL_REDACTED] | phone=[PHONE_REDACTED]", "client_ip_masked": "[IP_REDACTED]"}
{"timestamp": "2026-04-15T08:31:30Z", "ticket_id": "AS-1002", "service_type": "washing_machine", "status_code": 500, "status_label": "server_error", "event_text_masked": "ticket=AS-1002 | service=washing_machine | status=server_error | alt_phone=[PHONE_REDACTED]", "client_ip_masked": "[IP_REDACTED]"}
{"timestamp": "2026-04-15T08:33:40Z", "ticket_id": "AS-1003", "service_type": "refrigerator", "status_code": 404, "status_label": "client_error", "event_text_masked": "ticket=AS-1003 | service=refrigerator | status=client_error", "client_ip_masked": "[IP_REDACTED]"}


## 12) 품질 체크리스트 (학습자가 확인할 항목)

모든 셀 실행 후 다음을 확인하세요:

- [ ] 최종 출력에 유효하지 않은 timestamp가 없음
- [ ] `status_code`가 숫자이며 유효한 HTTP 범위에 있음
- [ ] 중복 레코드가 감소함
- [ ] PII 패턴이 마스킹(redacted)됨
- [ ] 출력이 유효한 JSONL 형식 (라인당 JSON 객체 1개)

선택 확장 아이디어:
1. 레코드마다 해시 기반 결정적(deterministic) ID 추가
2. 다국어 로그를 위한 언어 탐지 또는 텍스트 정규화 추가
3. 파서, 중복 제거 로직, PII 정규식(regex)에 대한 단위 테스트(unit test) 추가

## 13) 고급 실습 팩: 더 많은 사례, 더 많은 방법

이 섹션은 실무 데이터 품질 이슈를 추가하여 수업을 확장합니다.

새로운 시나리오:
1. 결측값(missing values) 및 스키마 드리프트(schema drift)
2. 불일치하는 범주형(categorical) 값 (예: service 별칭)
3. 동일 ticket에 대한 충돌(conflicting) 레코드
4. 숫자(numeric) 필드 이상치(outlier) 탐지
5. 혼합 형식에 대한 timestamp 표준화
6. 추가 형식 변환 예제 (XML -> JSONL)

목표는 **여러 정제(cleansing) 전략**을 비교하고 트레이드오프(trade-off)를 이해하는 것입니다.

In [14]:
# Build a deliberately messy A/S dataset for advanced practice.
advanced_raw = pd.DataFrame(
    [
        {
            "ticket_id": "as-2001",
            "service_type": "AirCon",
            "status_code": "200",
            "bytes_sent": "820",
            "timestamp": "2026-04-16 10:20:30+09:00",
            "agent_note": "Resolved. customer email: alpha.user@example.com",
            "source": "crm_csv",
        },
        {
            "ticket_id": "AS-2001",
            "service_type": "air conditioner",
            "status_code": 200,
            "bytes_sent": 820,
            "timestamp": "16/04/2026 10:20:30",
            "agent_note": "resolved",
            "source": "docx_export",
        },
        {
            "ticket_id": "AS-2002",
            "service_type": "washer",
            "status_code": "500",
            "bytes_sent": "9999999",
            "timestamp": "2026/04/16 10:22",
            "agent_note": "Tech requested callback 010-1111-2222",
            "source": "crm_csv",
        },
        {
            "ticket_id": "AS-2003",
            "service_type": "fridge",
            "status_code": "404",
            "bytes_sent": "-",
            "timestamp": "2026-04-16T01:00:00Z",
            "agent_note": "Part not found",
            "source": "web_form",
        },
        {
            "ticket_id": None,
            "service_type": "AC",
            "status_code": "200",
            "bytes_sent": "500",
            "timestamp": "bad_timestamp",
            "agent_note": "Missing ticket id",
            "source": "legacy",
        },
        {
            "ticket_id": "AS-2004",
            "service_type": None,
            "status_code": "301",
            "bytes_sent": "620",
            "timestamp": "2026-04-16 10:28:00+09:00",
            "agent_note": "redirected",
            "source": "legacy",
        },
        {
            "ticket_id": "AS-2005",
            "service_type": "washingmachine",
            "status_code": "two hundred",
            "bytes_sent": "700",
            "timestamp": "2026-04-16 10:30:00+09:00",
            "agent_note": "manual correction needed",
            "source": "manual_entry",
        },
    ]
)

print("고급 원본(raw) 샘플:")
display(advanced_raw)

고급 원본(raw) 샘플:


,ticket_id,service_type,status_code,bytes_sent,timestamp,agent_note,source
0,as-2001,AirCon,200,820,2026-04-16 10:20:30+09:00,Resolved. customer email: alpha.user@example.com,crm_csv
1,AS-2001,air conditioner,200,820,16/04/2026 10:20:30,resolved,docx_export
2,AS-2002,washer,500,9999999,2026/04/16 10:22,Tech requested callback 010-1111-2222,crm_csv
3,AS-2003,fridge,404,-,2026-04-16T01:00:00Z,Part not found,web_form
4,NaN,AC,200,500,bad_timestamp,Missing ticket id,legacy
5,AS-2004,NaN,301,620,2026-04-16 10:28:00+09:00,redirected,legacy
6,AS-2005,washingmachine,two hundred,700,2026-04-16 10:30:00+09:00,manual correction needed,manual_entry


## 14) 결측값(Missing Values) 처리: 방법 A vs 방법 B

단 하나의 "최선" 방법은 없습니다. 두 가지 일반적인 접근을 비교합니다:

- **방법 A (strict):** 핵심 키(`ticket_id`, `service_type`, `timestamp`)가 결측인 행 제거(drop)
- **방법 B (impute/fill):** placeholder를 대입(impute)하고 파싱 fallback을 사용해 행 유지

**데이터 품질 순도(data quality purity)**와 **데이터 보존(data retention)** 사이의 트레이드오프를 가르치는 데 사용합니다.

In [15]:
# Common pre-processing
adv = advanced_raw.copy()
adv["ticket_id"] = adv["ticket_id"].astype(str).str.upper().str.strip()
adv["ticket_id"] = adv["ticket_id"].replace({"NONE": pd.NA, "NAN": pd.NA})

# Flexible timestamp parser for mixed formats
adv["timestamp_parsed"] = pd.to_datetime(adv["timestamp"], errors="coerce", utc=True)
adv["status_code_num"] = pd.to_numeric(adv["status_code"], errors="coerce")
adv["bytes_sent_num"] = pd.to_numeric(adv["bytes_sent"].replace("-", pd.NA), errors="coerce")

# Method A: strict drop
method_a = adv.dropna(subset=["ticket_id", "service_type", "timestamp_parsed"]).copy()

# Method B: impute and keep
method_b = adv.copy()
method_b["ticket_id"] = method_b["ticket_id"].fillna("UNKNOWN_TICKET")
method_b["service_type"] = method_b["service_type"].fillna("unknown_service")
method_b["timestamp_parsed"] = method_b["timestamp_parsed"].fillna(pd.Timestamp("1970-01-01", tz="UTC"))

print("방법 A 행 수 (strict):", len(method_a))
print("방법 B 행 수 (impute):", len(method_b))

comparison = pd.DataFrame(
    {
        "metric": ["rows_kept", "missing_ticket_after", "missing_service_after", "missing_ts_after"],
        "method_a_strict": [
            len(method_a),
            method_a["ticket_id"].isna().sum(),
            method_a["service_type"].isna().sum(),
            method_a["timestamp_parsed"].isna().sum(),
        ],
        "method_b_impute": [
            len(method_b),
            method_b["ticket_id"].isna().sum(),
            method_b["service_type"].isna().sum(),
            method_b["timestamp_parsed"].isna().sum(),
        ],
    }
)
display(comparison)

방법 A 행 수 (strict): 2
방법 B 행 수 (impute): 7


,metric,method_a_strict,method_b_impute
0,rows_kept,2,7
1,missing_ticket_after,0,0
2,missing_service_after,0,0
3,missing_ts_after,0,0


## 15) 범주형(Categorical) 정규화(Normalization) 방법

범주(category) 열에 대한 두 가지 정규화 기법을 시연합니다:

1. **사전 매핑(dictionary mapping)** (빠르고 통제 가능)
2. **규칙 기반(rule-based) 정규화** — regex 및 토큰 정리 사용 (더 유연)

출력을 비교하고 어떤 방법이 유지보수에 더 쉬운지 판단할 수 있습니다.

In [16]:
cat_df = method_b.copy()

# Method 1: dictionary mapping
service_map_dict = {
    "aircon": "air_conditioner",
    "ac": "air_conditioner",
    "air conditioner": "air_conditioner",
    "washer": "washing_machine",
    "washingmachine": "washing_machine",
    "fridge": "refrigerator",
}

cat_df["service_dict_norm"] = (
    cat_df["service_type"].astype(str).str.lower().str.strip().map(service_map_dict).fillna("other")
)

# Method 2: rule-based normalization

def normalize_service_rule(x: str) -> str:
    t = str(x).lower().strip()
    t = re.sub(r"[^a-z ]", "", t)
    if re.search(r"\b(ac|aircon|air conditioner)\b", t):
        return "air_conditioner"
    if re.search(r"\b(washer|washing machine|washingmachine)\b", t):
        return "washing_machine"
    if re.search(r"\b(fridge|refrigerator)\b", t):
        return "refrigerator"
    if "unknown" in t:
        return "unknown_service"
    return "other"

cat_df["service_rule_norm"] = cat_df["service_type"].apply(normalize_service_rule)

print("카테고리 정규화 방법 비교:")
display(cat_df[["service_type", "service_dict_norm", "service_rule_norm"]])

카테고리 정규화 방법 비교:


,service_type,service_dict_norm,service_rule_norm
0,AirCon,air_conditioner,air_conditioner
1,air conditioner,air_conditioner,air_conditioner
2,washer,washing_machine,washing_machine
3,fridge,refrigerator,refrigerator
4,AC,air_conditioner,air_conditioner
5,unknown_service,other,unknown_service
6,washingmachine,washing_machine,washing_machine


## 16) 동일 Ticket에 대한 충돌(Conflict) 해결

동일 ticket이 서로 다른 시스템에서 다른 status로 여러 번 나타나는 경우가 있습니다.

두 가지 일반적인 규칙:
- **최신 이벤트 우선(latest-event wins)** (timestamp 기준)
- **우선순위 기반(priority-based wins)** (예: server_error > client_error > success)

데이터 정제(cleansing)에는 기술적 정리뿐 아니라 **비즈니스 규칙(business rules)**도 포함될 수 있음을 보여줍니다.

In [17]:
conflict_df = cat_df.copy()
conflict_df["status_code_num"] = pd.to_numeric(conflict_df["status_code_num"], errors="coerce")
conflict_df["status_label"] = pd.cut(
    conflict_df["status_code_num"],
    bins=[99, 299, 399, 499, 599],
    labels=["success", "redirect", "client_error", "server_error"],
)

# Rule 1: latest timestamp wins
latest_wins = (
    conflict_df.sort_values("timestamp_parsed")
    .drop_duplicates(subset=["ticket_id"], keep="last")
    [["ticket_id", "timestamp_parsed", "status_code_num", "status_label", "source"]]
    .sort_values("ticket_id")
)

# Rule 2: priority-based wins
priority_map = {"server_error": 4, "client_error": 3, "redirect": 2, "success": 1}
prio_df = conflict_df.copy()
prio_df["priority"] = prio_df["status_label"].astype(str).map(priority_map).fillna(0)
priority_wins = (
    prio_df.sort_values(["ticket_id", "priority", "timestamp_parsed"], ascending=[True, False, False])
    .drop_duplicates(subset=["ticket_id"], keep="first")
    [["ticket_id", "timestamp_parsed", "status_code_num", "status_label", "source", "priority"]]
    .sort_values("ticket_id")
)

print("최신 이벤트 우선:")
display(latest_wins)
print("우선순위 기반:")
display(priority_wins)

최신 이벤트 우선:


,ticket_id,timestamp_parsed,status_code_num,status_label,source
0,AS-2001,2026-04-16 01:20:30+00:00,200.0,success,crm_csv
2,AS-2002,1970-01-01 00:00:00+00:00,500.0,server_error,crm_csv
3,AS-2003,1970-01-01 00:00:00+00:00,404.0,client_error,web_form
5,AS-2004,2026-04-16 01:28:00+00:00,301.0,redirect,legacy
6,AS-2005,2026-04-16 01:30:00+00:00,NaN,NaN,manual_entry
4,UNKNOWN_TICKET,1970-01-01 00:00:00+00:00,200.0,success,legacy


우선순위 기반:


,ticket_id,timestamp_parsed,status_code_num,status_label,source,priority
0,AS-2001,2026-04-16 01:20:30+00:00,200.0,success,crm_csv,1.0
2,AS-2002,1970-01-01 00:00:00+00:00,500.0,server_error,crm_csv,4.0
3,AS-2003,1970-01-01 00:00:00+00:00,404.0,client_error,web_form,3.0
5,AS-2004,2026-04-16 01:28:00+00:00,301.0,redirect,legacy,2.0
6,AS-2005,2026-04-16 01:30:00+00:00,NaN,NaN,manual_entry,0.0
4,UNKNOWN_TICKET,1970-01-01 00:00:00+00:00,200.0,success,legacy,1.0


## 17) 숫자(Numeric) 열 이상치(Outlier) 처리

`bytes_sent` 같은 필드에 대해 두 가지 실용적인 방법을 시연합니다:

- **IQR capping (winsorization 스타일)**
- **Z-score filtering**

각 방법이 분포와 유지되는 행 수에 어떤 영향을 주는지 확인해야 합니다.

In [18]:
num_df = conflict_df.copy()
num_df = num_df[num_df["bytes_sent_num"].notna()].copy()

# Method 1: IQR capping
q1 = num_df["bytes_sent_num"].quantile(0.25)
q3 = num_df["bytes_sent_num"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
num_df["bytes_iqr_capped"] = num_df["bytes_sent_num"].clip(lower=lower, upper=upper)

# Method 2: Z-score filtering
mean_val = num_df["bytes_sent_num"].mean()
std_val = num_df["bytes_sent_num"].std(ddof=0)
if std_val == 0:
    num_df["z_score"] = 0.0
else:
    num_df["z_score"] = (num_df["bytes_sent_num"] - mean_val) / std_val

z_filtered = num_df[num_df["z_score"].abs() <= 3].copy()

print("IQR 범위:", (lower, upper))
print("z-filter 전 행 수:", len(num_df))
print("z-filter 후 행 수:", len(z_filtered))

display(num_df[["ticket_id", "bytes_sent_num", "bytes_iqr_capped", "z_score"]])

IQR 범위: (np.float64(370.0), np.float64(1090.0))
z-filter 전 행 수: 6
z-filter 후 행 수: 6


,ticket_id,bytes_sent_num,bytes_iqr_capped,z_score
0,AS-2001,820.0,820.0,-0.447179
1,AS-2001,820.0,820.0,-0.447179
2,AS-2002,9999999.0,1090.0,2.236068
4,UNKNOWN_TICKET,500.0,500.0,-0.447265
5,AS-2004,620.0,620.0,-0.447233
6,AS-2005,700.0,700.0,-0.447211


## 18) 추가 변환 예제: XML -> JSONL

CSV/DOCX를 넘어 변환 실습을 확장하기 위해, XML 이벤트 로그를 JSONL로 변환하는 예제입니다.

레거시(legacy) 시스템이나 미들웨어(middleware) export와 통합할 때 유용합니다.

In [19]:
import xml.etree.ElementTree as ET

xml_path = INTERIM_DIR / "as_logs.xml"
xml_jsonl_path = PROCESSED_DIR / "as_logs_from_xml.jsonl"

xml_text = """<logs>
  <log><ticket_id>AS-3001</ticket_id><service_type>ac</service_type><status_code>200</status_code><note>ok</note></log>
  <log><ticket_id>AS-3002</ticket_id><service_type>washer</service_type><status_code>500</status_code><note>call +82-10-5555-6666</note></log>
  <log><ticket_id>AS-3003</ticket_id><service_type>fridge</service_type><status_code>404</status_code><note>email support@sample.com</note></log>
</logs>"""
xml_path.write_text(xml_text, encoding="utf-8")

root = ET.fromstring(xml_text)
xml_records = []
for log in root.findall("log"):
    st = pd.to_numeric(log.findtext("status_code"), errors="coerce")
    xml_records.append(
        {
            "ticket_id": (log.findtext("ticket_id") or "").strip(),
            "service_type": (log.findtext("service_type") or "").strip(),
            "status_code": None if pd.isna(st) else int(st),
            "note": (log.findtext("note") or "").strip(),
            "source": "xml",
        }
    )

with xml_jsonl_path.open("w", encoding="utf-8") as f:
    for rec in xml_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("저장된 XML 원본:", xml_path)
print("XML에서 저장된 JSONL:", xml_jsonl_path)
print("레코드 수:", len(xml_records))

저장된 XML 원본: /home/ethan/newgen/KMOU_Course/Module_4/data/interim/as_logs.xml
XML에서 저장된 JSONL: /home/ethan/newgen/KMOU_Course/Module_4/data/processed/as_logs_from_xml.jsonl
레코드 수: 3


## 19) 데이터 품질 점수(Data Quality Scoring, 규칙 기반)

간단한 품질 점수(quality score)는 학습자가 출력 품질을 정량적으로 평가하도록 돕습니다.

예시 점수 로직 (0~100):
- ticket 형식이 유효하면 +25
- timestamp 파싱 가능하면 +25
- status code가 유효한 HTTP 범위이면 +25
- note/text에 명백한 PII가 남아 있지 않으면 +25

데이터 정제(cleansing) 결과를 측정 가능하게 만들고, 방법 간 비교를 쉽게 합니다.

In [20]:
score_df = pii.copy()

ticket_ok_re = re.compile(r"^AS-\d{4,}$")

# Reuse PII patterns from earlier section if present.
if "email_re" not in globals():
    email_re = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
if "phone_re" not in globals():
    phone_re = re.compile(r"(?:\+?\d{1,3}[\s-]?)?(?:\d{2,4}[\s-]?){2,4}\d{3,4}")


def quality_score_row(row):
    score = 0

    ticket = str(row.get("ticket_id", ""))
    ts = row.get("timestamp")
    status = row.get("status_code")
    text = str(row.get("event_text_masked", row.get("event_text_normalized", "")))

    if ticket_ok_re.match(ticket):
        score += 25
    if pd.notna(ts):
        score += 25
    if pd.notna(status) and 100 <= float(status) <= 599:
        score += 25

    has_pii = bool(email_re.search(text) or phone_re.search(text))
    if not has_pii:
        score += 25

    return score

score_df["quality_score"] = score_df.apply(quality_score_row, axis=1)

print("품질 점수 분포:")
display(score_df["quality_score"].value_counts().sort_index())
print("평균 점수:", round(score_df["quality_score"].mean(), 2))
display(score_df[["ticket_id", "status_code", "event_text_masked", "quality_score"]].head(10))

품질 점수 분포:


quality_score
100    9
Name: count, dtype: int64

평균 점수: 100.0


,ticket_id,status_code,event_text_masked,quality_score
0,AS-1001,200,ticket=AS-1001 | service=air_conditioner | status=success | contact=[EMAIL_REDACTED] | phone=[PHONE_REDACTED],100
1,AS-1002,500,ticket=AS-1002 | service=washing_machine | status=server_error | alt_phone=[PHONE_REDACTED],100
3,AS-1003,404,ticket=AS-1003 | service=refrigerator | status=client_error,100
4,AS-1003,200,ticket=AS-1003 | service=refrigerator | status=success,100
5,AS-1004,200,ticket=AS-1004 | service=air_conditioner | status=success,100
6,AS-1001,200,nan,100
7,AS-1002,500,nan,100
9,AS-1003,404,nan,100
11,AS-1004,200,nan,100


## 20) 권장 학습자 연습 문제 (Hands-on)

1. 노이즈가 많은 레코드 10개 이상 추가 (결측 필드, 오타 카테고리, 잘못된 status code, 혼합 timezone).
2. strict vs imputation 전략을 비교하고 각각이 언제 더 적합한지 설명.
3. 변환 경로 하나 더 추가: `TSV -> JSONL`.
4. 근사 중복(near-duplicate) 로직을 유사도 매칭(similarity matching)으로 개선 (예: `difflib.SequenceMatcher`).
5. 주민등록번호(resident ID) 또는 신용카드 형태 패턴에 대한 PII 마스킹 확장.
6. 미니 벤치마크 표 작성: 유지된 행 수, 제거된 중복, 발견된 PII 누출, 평균 quality score.

이 연습은 워크샵을 더 풍부하게 만들고 실제 프로덕션 정제(cleansing) 파이프라인에 더 가깝게 만듭니다.